In [1]:
import bacco
import numpy as np

In [6]:
snap = 265

basedir = "/cosmos_storage/simulations/TNG_Family/MTNG/DM-Gadget4/MTNG-L500-2160-A"

# Load MTNG
dm_mtng = bacco.Simulation(basedir=basedir, halo_file="groups_%03d/fof_subhalo_tab_%03d"%(snap,snap),
                        sim_format='TNG_onlyDM', dm_file="snapdir_%03d/snapshot_%03d"%(snap,snap))

2026-07-08 23:32:05,481 bacco.sims : Initialising simulation Default
2026-07-08 23:32:05,483 bacco.sims : try /cosmos_storage/simulations/TNG_Family/MTNG/DM-Gadget4/MTNG-L500-2160-A/snapdir_265/snapshot_265
2026-07-08 23:32:05,525 bacco.sims : Loading /cosmos_storage/simulations/TNG_Family/MTNG/DM-Gadget4/MTNG-L500-2160-A/snapdir_265/snapshot_265
2026-07-08 23:32:05,541 bacco.sims : ...done in 0.0597 s


In [7]:
# Load halo selection
with open("/cosmos_storage/home/fgmaion/MTNG-resims/halo_selection/dm_halo_sel_1pmbin.txt") as f:
    dm_sel = []
    for line in f.readlines():
        dm_sel.append(int(line.split()[0]))
dm_sel = np.array(dm_sel)


In [15]:
part = dm_mtng.get_halo_particles(dm_sel[0])

2026-07-08 23:36:53,914 bacco.sims : Reading 11273943 items for Group/GroupLen
2026-07-08 23:36:56,284 bacco.sims : Reading 11273943 items for GroupLen
2026-07-08 23:36:56,838 bacco.sims : Read data for 172227489/10077696000 particles...


/cosmos_storage/simulations/TNG_Family/MTNG/DM-Gadget4/MTNG-L500-2160-A/snapdir_265/snapshot_265.0.hdf5


TypeError: data type '<u6' not understood

In [ ]:
# Load halo selection
with open("/cosmos_storage/home/fgmaion/MTNG-resims/halo_selection/hydro_halo_sel_1pmbin.txt") as f:
    final_sel = []
    for line in f.readlines():
        final_sel.append(int(line.split()[0]))
final_sel = np.array(final_sel)

# position matching
X1 = zoom.fof['halo_pos']
X2 = mtng.fof['halo_pos'][final_sel]

kdt = scipy.spatial.KDTree(X1, boxsize=mtng.header['BoxSize'])
dist, ind = kdt.query(X2, k=100)

# load halo properties
M_zoom = 1e10 * zoom.fof['halo_m200b'][ind]
M_mtng = 1e10 * mtng.fof['halo_m200b'][final_sel]

v_zoom = zoom.fof['halo_vel'][ind,:]
v_mtng = mtng.fof['halo_vel'][final_sel,:]

cos = np.sum(v_zoom * v_mtng[:,np.newaxis,:], axis=2) / ( np.linalg.norm(v_zoom, axis=2) * np.linalg.norm(v_mtng, axis=1)[:,np.newaxis] )
v_ratio = np.linalg.norm(v_zoom, axis=2) / np.linalg.norm(v_mtng, axis=1)[:,np.newaxis]

d = metric(M_mtng, M_zoom, cos, v_ratio, dist, 1, 1, 1, 1)

xmatch = np.zeros(len(M_zoom), dtype=int)
dmatch = np.zeros(len(M_zoom))
metr = np.zeros(len(M_zoom))

for i in range(len(final_sel)):
    metr[i] = d[i].min()
    xmatch[i] = ind[i,np.where(d[i]==metr[i])[0][0]]
    dmatch[i] = dist[i,np.where(d[i]==metr[i])[0][0]]